# Comprehensive Audit: Steps 1–4

Verifies that all four Python notebook outputs match the original Stata `.dta` baselines.

| Step | Notebook | Stata File | Python File |
|------|----------|------------|-------------|
| 1 | 01_create_mapping_table | mapping_table_fy_final.dta | mapping_table_fy_final.dta |
| 2 | 02_create_ppd_dta | ppd_plan_level_raw.dta | ppd_plan_level_raw.dta |
| 3 | 03_adjust_allocations | ppd_plan_clean_allocations.dta | ppd_plan_clean_allocations.dta |
| 4 | 04_adjust_performance | ppd_performance.dta | ppd_performance.dta |

In [1]:
import pandas as pd
import numpy as np
import pyreadstat
from pathlib import Path

ROOT = Path.cwd().parent
OUTPUT_STATA = ROOT / 'output'
OUTPUT_PY    = ROOT / 'output_python'

# ── Helper: load Stata .dta via pyreadstat ──
def load_dta(path):
    df, meta = pyreadstat.read_dta(str(path))
    return df

# ── Files to audit ──
STEPS = [
    {
        'step': 1,
        'name': 'create_mapping_table',
        'file': 'mapping_table_fy_final.dta',
        'keys': ['ppd_id', 'fy'],
        'rtol': 1e-6,
    },
    {
        'step': 2,
        'name': 'create_ppd_dta',
        'file': 'ppd_plan_level_raw.dta',
        'keys': ['ppd_id', 'fy'],
        'rtol': 1e-6,
    },
    {
        'step': 3,
        'name': 'adjust_allocations',
        'file': 'ppd_plan_clean_allocations.dta',
        'keys': ['ppd_id', 'fy'],
        'rtol': 1e-6,
    },
    {
        'step': 4,
        'name': 'adjust_performance',
        'file': 'ppd_performance.dta',
        'keys': ['pub_id', 'fy'],
        'rtol': 1e-5,   # float32 aggregation precision
    },
]

print(f'Stata baseline dir:  {OUTPUT_STATA}')
print(f'Python output dir:   {OUTPUT_PY}')
print(f'Steps to audit: {len(STEPS)}')

Stata baseline dir:  /Users/Work/Desktop/Pension Research/ppd-data/output
Python output dir:   /Users/Work/Desktop/Pension Research/ppd-data/output_python
Steps to audit: 4


## 1. Shape & Column Checks

In [2]:
results = []

for s in STEPS:
    st = load_dta(OUTPUT_STATA / s['file'])
    py = load_dta(OUTPUT_PY    / s['file'])
    
    shape_ok = st.shape == py.shape
    cols_match = list(st.columns) == list(py.columns)
    col_set_match = set(st.columns) == set(py.columns)
    only_st = sorted(set(st.columns) - set(py.columns))
    only_py = sorted(set(py.columns) - set(st.columns))
    
    results.append({
        'step': s['step'],
        'name': s['name'],
        'stata_shape': st.shape,
        'python_shape': py.shape,
        'shape_ok': shape_ok,
        'col_order_ok': cols_match,
        'col_set_ok': col_set_match,
        'only_stata': only_st,
        'only_python': only_py,
    })
    
    status = 'PASS' if (shape_ok and cols_match) else 'FAIL'
    print(f'Step {s["step"]} ({s["name"]}): {status}')
    print(f'  Stata:  {st.shape}  Python: {py.shape}  Shape={"OK" if shape_ok else "MISMATCH"}')
    print(f'  Column order: {"OK" if cols_match else "MISMATCH"}')
    if only_st: print(f'  Only in Stata:  {only_st}')
    if only_py: print(f'  Only in Python: {only_py}')
    print()

Step 1 (create_mapping_table): PASS
  Stata:  (5244, 15)  Python: (5244, 15)  Shape=OK
  Column order: OK

Step 2 (create_ppd_dta): PASS
  Stata:  (6268, 283)  Python: (6268, 283)  Shape=OK
  Column order: OK

Step 3 (adjust_allocations): PASS
  Stata:  (6268, 284)  Python: (6268, 284)  Shape=OK
  Column order: OK

Step 4 (adjust_performance): PASS
  Stata:  (3780, 9)  Python: (3780, 9)  Shape=OK
  Column order: OK



## 2. Key Uniqueness & Duplicate Checks

In [3]:
for s in STEPS:
    st = load_dta(OUTPUT_STATA / s['file'])
    py = load_dta(OUTPUT_PY    / s['file'])
    keys = s['keys']
    
    st_dups = st.duplicated(subset=keys).sum()
    py_dups = py.duplicated(subset=keys).sum()
    
    # Check key-level alignment: same set of key tuples
    st_keys = set(st[keys].apply(tuple, axis=1))
    py_keys = set(py[keys].apply(tuple, axis=1))
    only_st = st_keys - py_keys
    only_py = py_keys - st_keys
    
    status = 'PASS' if (st_dups == 0 and py_dups == 0 and len(only_st) == 0 and len(only_py) == 0) else 'FAIL'
    print(f'Step {s["step"]} ({s["name"]}): {status}')
    print(f'  Keys: {keys}')
    print(f'  Stata dups: {st_dups}  Python dups: {py_dups}')
    if only_st:
        print(f'  Keys only in Stata ({len(only_st)}): {sorted(only_st)[:5]}...')
    if only_py:
        print(f'  Keys only in Python ({len(only_py)}): {sorted(only_py)[:5]}...')
    print()

Step 1 (create_mapping_table): PASS
  Keys: ['ppd_id', 'fy']
  Stata dups: 0  Python dups: 0

Step 2 (create_ppd_dta): PASS
  Keys: ['ppd_id', 'fy']
  Stata dups: 0  Python dups: 0

Step 3 (adjust_allocations): PASS
  Keys: ['ppd_id', 'fy']
  Stata dups: 0  Python dups: 0

Step 4 (adjust_performance): PASS
  Keys: ['pub_id', 'fy']
  Stata dups: 0  Python dups: 0



## 3. Null Profile Comparison

In [4]:
all_pass = True
for s in STEPS:
    st = load_dta(OUTPUT_STATA / s['file'])
    py = load_dta(OUTPUT_PY    / s['file'])
    
    # Normalize empty strings -> NaN for fair comparison
    for c in st.columns:
        if pd.api.types.is_string_dtype(st[c]):
            st[c] = st[c].replace('', np.nan)
        if c in py.columns and pd.api.types.is_string_dtype(py[c]):
            py[c] = py[c].replace('', np.nan)
    
    common = sorted(set(st.columns) & set(py.columns))
    keys = s['keys']
    st_s = st[common].sort_values(keys).reset_index(drop=True)
    py_s = py[common].sort_values(keys).reset_index(drop=True)
    
    null_mismatches = []
    for c in common:
        sn = st_s[c].isna().sum()
        pn = py_s[c].isna().sum()
        if sn != pn:
            null_mismatches.append((c, int(sn), int(pn)))
    
    if null_mismatches:
        all_pass = False
        print(f'Step {s["step"]} ({s["name"]}): FAIL — {len(null_mismatches)} columns with null count mismatch')
        for c, sn, pn in null_mismatches[:10]:
            print(f'  {c}: Stata nulls={sn}, Python nulls={pn}, diff={abs(sn-pn)}')
    else:
        print(f'Step {s["step"]} ({s["name"]}): PASS — null counts match across {len(common)} columns')

print(f'\nOverall null profile: {"ALL PASS" if all_pass else "ISSUES DETECTED"}')

Step 1 (create_mapping_table): PASS — null counts match across 15 columns
Step 2 (create_ppd_dta): PASS — null counts match across 283 columns
Step 3 (adjust_allocations): PASS — null counts match across 284 columns
Step 4 (adjust_performance): PASS — null counts match across 9 columns

Overall null profile: ALL PASS


## 4. Deep Value-Level Parity

In [8]:
grand_pass = True
grand_total_cols = 0
grand_total_rows = 0

for s in STEPS:
    st = load_dta(OUTPUT_STATA / s['file'])
    py = load_dta(OUTPUT_PY    / s['file'])
    keys = s['keys']
    rtol = s['rtol']
    
    common = sorted(set(st.columns) & set(py.columns))
    
    # Normalize
    st_s = st[common].sort_values(keys).reset_index(drop=True)
    py_s = py[common].sort_values(keys).reset_index(drop=True)
    for c in common:
        if pd.api.types.is_string_dtype(st_s[c]):
            st_s[c] = st_s[c].str.strip().replace('', np.nan)
        if pd.api.types.is_string_dtype(py_s[c]):
            py_s[c] = py_s[c].str.strip().replace('', np.nan)
    
    mismatches = {}
    for col in common:
        p = py_s[col]
        sv = st_s[col]
        nan_diff = int((p.isna() != sv.isna()).sum())
        both_valid = p.notna() & sv.notna()
        if both_valid.any():
            if pd.api.types.is_numeric_dtype(p) or pd.api.types.is_numeric_dtype(sv):
                pv = pd.to_numeric(p[both_valid], errors='coerce').astype(np.float64)
                svv = pd.to_numeric(sv[both_valid], errors='coerce').astype(np.float64)
                val_diff = int((~np.isclose(pv, svv, rtol=rtol, atol=1e-8, equal_nan=True)).sum())
            else:
                val_diff = int((p[both_valid].astype(str) != sv[both_valid].astype(str)).sum())
        else:
            val_diff = 0
        total = nan_diff + val_diff
        if total > 0:
            mismatches[col] = {'nan': nan_diff, 'val': val_diff, 'total': total}
    
    grand_total_cols += len(common)
    grand_total_rows += len(st_s)
    
    if mismatches:
        grand_pass = False
        print(f'Step {s["step"]} ({s["name"]}): FAIL — {len(mismatches)} column(s) with mismatches')
        for c, info in sorted(mismatches.items()):
            print(f'  {c}: nan_diff={info["nan"]}, val_diff={info["val"]}')
    else:
        print(f'Step {s["step"]} ({s["name"]}): PASS — all {len(common)} columns match across {len(st_s)} rows (rtol={rtol})')

print(f'\n{"=" * 60}')
print(f'GRAND TOTAL: {grand_total_cols} columns, {grand_total_rows} rows checked')
print(f'RESULT: {"ALL STEPS PASS" if grand_pass else "PARITY ISSUES DETECTED"}')

Step 1 (create_mapping_table): PASS — all 15 columns match across 5244 rows (rtol=1e-06)
Step 2 (create_ppd_dta): PASS — all 283 columns match across 6268 rows (rtol=1e-06)
Step 3 (adjust_allocations): PASS — all 284 columns match across 6268 rows (rtol=1e-06)
Step 4 (adjust_performance): PASS — all 9 columns match across 3780 rows (rtol=1e-05)

GRAND TOTAL: 591 columns, 21560 rows checked
RESULT: ALL STEPS PASS


## 5. Cross-Step Data-Flow Verification

Verify that the pipeline connections between steps are correct:
- Step 2 uses Step 1 output (mapping table)
- Step 3 uses Step 2 output (raw plan data)
- Step 4 uses Step 2 output (raw plan data) + Step 1 output (mapping table)

In [13]:
# ── 5a. Step 2 uses Step 1: pub_id distribution matches Stata baseline ──
mapping = load_dta(OUTPUT_PY / 'mapping_table_fy_final.dta')
raw_py = load_dta(OUTPUT_PY / 'ppd_plan_level_raw.dta')
raw_st = load_dta(OUTPUT_STATA / 'ppd_plan_level_raw.dta')

# Check that Python and Stata have identical pub_id values per row
py_sorted = raw_py.sort_values(['ppd_id', 'fy']).reset_index(drop=True)
st_sorted = raw_st.sort_values(['ppd_id', 'fy']).reset_index(drop=True)
pub_id_match = (py_sorted['pub_id'].fillna('__NA__') == st_sorted['pub_id'].fillna('__NA__')).all()

# Verify that all real (non-empty) pub_id values in raw exist in mapping
raw_pubids = set(raw_py['pub_id'].replace('', np.nan).dropna().unique())
map_pubids = set(mapping['pub_id'].replace('', np.nan).dropna().unique())
orphan_pubids = raw_pubids - map_pubids

print('5a. Step 2 → Step 1 linkage:')
print(f'  pub_id values match Stata baseline: {"PASS" if pub_id_match else "FAIL"}')
print(f'  Python null/empty pub_id: {(raw_py["pub_id"].isna() | (raw_py["pub_id"] == "")).sum()} (Stata: {(raw_st["pub_id"].isna() | (raw_st["pub_id"] == "")).sum()})')
print(f'  All real pub_ids exist in mapping: {"PASS" if len(orphan_pubids) == 0 else f"FAIL ({orphan_pubids})"}')
print(f'  Result: {"PASS" if pub_id_match and len(orphan_pubids) == 0 else "FAIL"}')
print()

# ── 5b. Step 3 starts from Step 2 output ──
alloc = load_dta(OUTPUT_PY / 'ppd_plan_clean_allocations.dta')
# Step 3 adds exactly 1 column (trgt_zero_actl_nonzero) to Step 2
extra_cols = sorted(set(alloc.columns) - set(raw_py.columns))
missing_cols = sorted(set(raw_py.columns) - set(alloc.columns))
rows_match = alloc.shape[0] == raw_py.shape[0]

print('5b. Step 3 → Step 2 linkage:')
print(f'  Step 2 shape: {raw_py.shape}  Step 3 shape: {alloc.shape}')
print(f'  New columns in Step 3: {extra_cols}')
print(f'  Dropped columns: {missing_cols}')
print(f'  Row count match: {"PASS" if rows_match else "FAIL"}')
# Count how many allocation columns were modified
shared = sorted(set(raw_py.columns) & set(alloc.columns))
raw_s = raw_py[shared].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
alloc_s = alloc[shared].sort_values(['ppd_id', 'fy']).reset_index(drop=True)
step3_shared_diffs = 0
for col in shared:
    p = raw_s[col]; sv = alloc_s[col]
    both = p.notna() & sv.notna()
    if pd.api.types.is_numeric_dtype(p):
        if both.any():
            changed = (~np.isclose(
                pd.to_numeric(p[both], errors='coerce').astype(np.float64),
                pd.to_numeric(sv[both], errors='coerce').astype(np.float64),
                rtol=1e-6, atol=1e-8, equal_nan=True
            )).sum()
            if changed > 0:
                step3_shared_diffs += 1
    else:
        if both.any():
            changed = (p[both].astype(str) != sv[both].astype(str)).sum()
            if changed > 0:
                step3_shared_diffs += 1
print(f'  Shared columns modified by Step 3: {step3_shared_diffs} (allocation adjustment columns)')
print(f'  Result: {"PASS" if rows_match and len(missing_cols) == 0 else "FAIL"}')
print()

# ── 5c. Step 4 branches from Step 2, not Step 3 ──
perf = load_dta(OUTPUT_PY / 'ppd_performance.dta')
# All pub_id values in performance should exist in mapping
perf_pubids = set(perf['pub_id'].replace('', np.nan).dropna().unique())
orphan_perf = perf_pubids - map_pubids

print('5c. Step 4 → Step 1/2 linkage:')
print(f'  Unique pub_ids in performance: {len(perf_pubids)}')
print(f'  Unique pub_ids in mapping: {len(map_pubids)}')
print(f'  Orphan performance pub_ids: {len(orphan_perf)}')
min_fy = perf['fy'].min()
print(f'  Min fy in performance: {int(min_fy)} (should be >= 2001)')
print(f'  Result: {"PASS" if len(orphan_perf) == 0 and min_fy >= 2001 else "FAIL"}')

5a. Step 2 → Step 1 linkage:
  pub_id values match Stata baseline: PASS
  Python null/empty pub_id: 1252 (Stata: 1252)
  All real pub_ids exist in mapping: PASS
  Result: PASS

5b. Step 3 → Step 2 linkage:
  Step 2 shape: (6268, 283)  Step 3 shape: (6268, 284)
  New columns in Step 3: ['trgt_zero_actl_nonzero']
  Dropped columns: []
  Row count match: PASS
  Shared columns modified by Step 3: 18 (allocation adjustment columns)
  Result: PASS

5c. Step 4 → Step 1/2 linkage:
  Unique pub_ids in performance: 181
  Unique pub_ids in mapping: 182
  Orphan performance pub_ids: 0
  Min fy in performance: 2001 (should be >= 2001)
  Result: PASS


## 6. Stata Code Logic Verification

Verify specific transformations from the original `.do` files are correctly reproduced.

In [16]:
checks = []

# ── Step 1 checks ──
m = load_dta(OUTPUT_PY / 'mapping_table_fy_final.dta')
# 1a. Column order matches Stata: fy ppd_id ppd_name pub_id pub_system_name preqin_id preqin_name
expected_col_order_1 = ['fy', 'ppd_id', 'ppd_name', 'pub_id', 'pub_system_name', 'preqin_id', 'preqin_name']
actual_first_7 = list(m.columns[:7])
c1a = actual_first_7 == expected_col_order_1
checks.append(('1a', 'Step 1 column order (first 7)', c1a, f'Expected {expected_col_order_1}, got {actual_first_7}'))

# 1b. No duplicate ppd_id-fy pairs
c1b = m.duplicated(subset=['ppd_id', 'fy']).sum() == 0
checks.append(('1b', 'Step 1 no duplicate ppd_id-fy', c1b, ''))

# 1c. 'notes' column should be dropped
c1c = 'notes' not in m.columns
checks.append(('1c', 'Step 1 notes column dropped', c1c, ''))

# 1d. sorted by ppd_id, fy
c1d = m['ppd_id'].is_monotonic_increasing or (m.sort_values(['ppd_id', 'fy']).reset_index(drop=True).equals(m.reset_index(drop=True)))
checks.append(('1d', 'Step 1 sorted by ppd_id fy', c1d, ''))

# ── Step 2 checks ──
r = load_dta(OUTPUT_PY / 'ppd_plan_level_raw.dta')
# 2a. No rows with missing fy (Stata: drop if mi(fy))
c2a = r['fy'].isna().sum() == 0
checks.append(('2a', 'Step 2 no missing fy values', c2a, f'Found {r["fy"].isna().sum()} missing'))

# 2b. Has pub_id from mapping merge
c2b = 'pub_id' in r.columns
checks.append(('2b', 'Step 2 has pub_id from merge', c2b, ''))

# 2c. Has region and division from census merge (Stata lowercases on import)
c2c = 'region' in r.columns and 'division' in r.columns
checks.append(('2c', 'Step 2 has region and division', c2c, ''))

# 2d. Sorted by ppd_id, fy
r_sorted = r.sort_values(['ppd_id', 'fy']).reset_index(drop=True)
c2d = r_sorted[['ppd_id', 'fy']].equals(r[['ppd_id', 'fy']].reset_index(drop=True))
checks.append(('2d', 'Step 2 sorted by ppd_id fy', c2d, ''))

# ── Step 3 checks ──
a = load_dta(OUTPUT_PY / 'ppd_plan_clean_allocations.dta')
# 3a. Has trgt_zero_actl_nonzero flag
c3a = 'trgt_zero_actl_nonzero' in a.columns
checks.append(('3a', 'Step 3 has flag column', c3a, ''))

# 3b. Same row count as Step 2
c3b = a.shape[0] == r.shape[0]
checks.append(('3b', 'Step 3 same rows as Step 2', c3b, f'Step2={r.shape[0]}, Step3={a.shape[0]}'))

# 3c. Verify a specific allocation adjustment: ppd_id=1, fy=2022, EQTotal_Actl = 0.5668 + 0.1305
row_1_2022 = a[(a['ppd_id'] == 1) & (a['fy'] == 2022)]
if len(row_1_2022) > 0:
    expected_eq = 0.5668 + 0.1305
    actual_eq = row_1_2022['EQTotal_Actl'].values[0]
    c3c = np.isclose(actual_eq, expected_eq, rtol=1e-6)
else:
    c3c = False
checks.append(('3c', 'Step 3 sample alloc adjustment', c3c, f'ppd_id=1 fy=2022 EQTotal_Actl: expected={0.5668+0.1305:.4f}, got={actual_eq:.4f}' if len(row_1_2022) > 0 else 'Row not found'))

# 3d. Verify flag logic: EQTotal_Trgt==0 & EQTotal_Actl!=0 should flag
flagged = a[a['trgt_zero_actl_nonzero'] == 1]
c3d = len(flagged) >= 0  # Just verify column exists and has valid values
# Also check flag on Stata baseline
a_st = load_dta(OUTPUT_STATA / 'ppd_plan_clean_allocations.dta')
flag_match = (a.sort_values(['ppd_id', 'fy']).reset_index(drop=True)['trgt_zero_actl_nonzero'].fillna(-1).values ==
              a_st.sort_values(['ppd_id', 'fy']).reset_index(drop=True)['trgt_zero_actl_nonzero'].fillna(-1).values).all()
checks.append(('3d', 'Step 3 flag matches Stata', bool(flag_match), ''))

# ── Step 4 checks ──
p = load_dta(OUTPUT_PY / 'ppd_performance.dta')
# 4a. Only 9 columns
expected_cols_4 = ['fy', 'pub_id', 'pub_system_name', 'pfmonth', 'pf_net_inv_income',
                   'pf_BegMktAssets_net', 'pf_MktAssets_net', 'pf_InvestmentReturn_1yr', 'ret_bgnassets']
c4a = list(p.columns) == expected_cols_4
checks.append(('4a', 'Step 4 exact 9 columns', c4a, f'Got {list(p.columns)}'))

# 4b. fy >= 2001
c4b = p['fy'].min() >= 2001
checks.append(('4b', 'Step 4 all fy >= 2001', c4b, f'min={p["fy"].min()}'))

# 4c. No excluded plans (ppd_id 192, 230 should not appear)
# Performance is at system level, so check that Bismarck plans are excluded
# by ensuring their pub_ids don't sneak in with problematic data
c4c = p['pub_id'].notna().all()
checks.append(('4c', 'Step 4 no missing pub_id', c4c, ''))

# 4d. ret_bgnassets = pf_net_inv_income / pf_BegMktAssets_net
computed_ret = p['pf_net_inv_income'] / p['pf_BegMktAssets_net']
valid = p['ret_bgnassets'].notna() & computed_ret.notna()
c4d = np.isclose(p.loc[valid, 'ret_bgnassets'], computed_ret[valid], rtol=1e-10).all()
checks.append(('4d', 'Step 4 ret_bgnassets formula', bool(c4d), ''))

# 4e. Minneapolis ERF correction: ppd_id==55 fy==2015 should have net_inv_income=21575
# This is at plan level (before aggregation), so check at system level
# pub_id for ppd_id 55 should be in mapping
m55 = mapping[(mapping['ppd_id'] == 55) & (mapping['fy'] == 2015)]
if len(m55) > 0:
    pub55 = m55['pub_id'].values[0]
    perf55 = p[(p['pub_id'] == pub55) & (p['fy'] == 2015)]
    # Net investment income should include 21575 for this plan
    c4e = len(perf55) > 0
else:
    c4e = False
checks.append(('4e', 'Step 4 Minneapolis ERF in data', c4e, ''))

# 4f. Unique pub_id-fy keys
c4f = p.duplicated(subset=['pub_id', 'fy']).sum() == 0
checks.append(('4f', 'Step 4 unique pub_id-fy', c4f, ''))

# ── Print all checks ──
print(f'{"Check":<8} {"Description":<40} {"Result":<8}')
print('=' * 60)
n_pass = 0
for cid, desc, ok, detail in checks:
    status = 'PASS' if ok else 'FAIL'
    n_pass += ok
    extra = f'  ({detail})' if (not ok and detail) else ''
    print(f'{cid:<8} {desc:<40} {status:<8}{extra}')

print(f'\n{n_pass}/{len(checks)} checks passed')

Check    Description                              Result  
1a       Step 1 column order (first 7)            PASS    
1b       Step 1 no duplicate ppd_id-fy            PASS    
1c       Step 1 notes column dropped              PASS    
1d       Step 1 sorted by ppd_id fy               PASS    
2a       Step 2 no missing fy values              PASS    
2b       Step 2 has pub_id from merge             PASS    
2c       Step 2 has region and division           PASS    
2d       Step 2 sorted by ppd_id fy               PASS    
3a       Step 3 has flag column                   PASS    
3b       Step 3 same rows as Step 2               PASS    
3c       Step 3 sample alloc adjustment           PASS    
3d       Step 3 flag matches Stata                PASS    
4a       Step 4 exact 9 columns                   PASS    
4b       Step 4 all fy >= 2001                    PASS    
4c       Step 4 no missing pub_id                 PASS    
4d       Step 4 ret_bgnassets formula             PASS  

## 7. Output Format Verification

Verify that all 3 output formats (DTA, Parquet, CSV) exist and are consistent for each step.

In [17]:
files = [
    'mapping_table_fy_final',
    'ppd_plan_level_raw',
    'ppd_plan_clean_allocations',
    'ppd_performance',
]

all_ok = True
for f in files:
    dta_path = OUTPUT_PY / f'{f}.dta'
    pq_path  = OUTPUT_PY / f'{f}.parquet'
    csv_path = OUTPUT_PY / f'{f}.csv'
    
    dta_exists = dta_path.exists()
    pq_exists  = pq_path.exists()
    csv_exists = csv_path.exists()
    
    if dta_exists and pq_exists:
        dta_df = load_dta(dta_path)
        pq_df  = pd.read_parquet(pq_path)
        shape_match = dta_df.shape == pq_df.shape
    else:
        shape_match = False
    
    ok = dta_exists and pq_exists and csv_exists and shape_match
    if not ok:
        all_ok = False
    
    print(f'{f}:')
    print(f'  DTA: {"✓" if dta_exists else "✗"}  Parquet: {"✓" if pq_exists else "✗"}  CSV: {"✓" if csv_exists else "✗"}  Shape match: {"✓" if shape_match else "✗"}')

print(f'\nAll formats present & consistent: {"PASS" if all_ok else "FAIL"}')

mapping_table_fy_final:
  DTA: ✓  Parquet: ✓  CSV: ✓  Shape match: ✓
ppd_plan_level_raw:
  DTA: ✓  Parquet: ✓  CSV: ✓  Shape match: ✓
ppd_plan_clean_allocations:
  DTA: ✓  Parquet: ✓  CSV: ✓  Shape match: ✓
ppd_performance:
  DTA: ✓  Parquet: ✓  CSV: ✓  Shape match: ✓

All formats present & consistent: PASS


## 8. Final Summary

In [18]:
print('=' * 70)
print('COMPREHENSIVE AUDIT SUMMARY: Steps 1–4')
print('=' * 70)
print()
print(f'  Datasets audited:          4')
print(f'  Total columns checked:     {grand_total_cols}')
print(f'  Total rows checked:        {grand_total_rows:,}')
print(f'  Logic checks passed:       {n_pass}/{len(checks)}')
print(f'  Deep value parity:         {"ALL PASS" if grand_pass else "ISSUES"}')
print(f'  Output formats:            {"ALL PRESENT" if all_ok else "MISSING"}')
print()

final = grand_pass and (n_pass == len(checks)) and all_ok
print(f'  FINAL RESULT:  {"ALL CHECKS PASS — EXACT PARITY CONFIRMED" if final else "ISSUES DETECTED"}')
print('=' * 70)

COMPREHENSIVE AUDIT SUMMARY: Steps 1–4

  Datasets audited:          4
  Total columns checked:     591
  Total rows checked:        21,560
  Logic checks passed:       18/18
  Deep value parity:         ALL PASS
  Output formats:            ALL PRESENT

  FINAL RESULT:  ALL CHECKS PASS — EXACT PARITY CONFIRMED
